# Part 3: Báo Cáo Kỹ Nghệ Đặc Trưng (Feature Engineering)
**Thành viên phụ trách:** Thành viên 3 | **Nhánh Git:** `feature/feature-engineering`
---
Notebook trích xuất 22 đặc trưng mới: chu kỳ thời gian sin-cos, chỉ số nguy cơ thời tiết, tải sân bay.

### 1. Trích Xuất Toàn Bộ Đặc Trưng Mới


In [ ]:
import pandas as pd
from config import FILE_CLEANED
from features.features import build_engineered_features

raw_df = pd.read_csv(FILE_CLEANED)
feat_df = build_engineered_features(raw_df)
print(f'Số cột ban đầu: {raw_df.shape[1]} -> Số cột sau kỹ nghệ: {feat_df.shape[1]} (+{feat_df.shape[1] - raw_df.shape[1]} đặc trưng)')
display(feat_df[['scheduled_time', 'hour_sin', 'hour_cos', 'weather_hazard_score', 'is_peak_hour', 'load_x_weather']].head(5))


### 2. Tương Quan Giữa Nguy Cơ Thời Tiết Và Rủi Ro Trễ


In [ ]:
import matplotlib.pyplot as plt
feat_df['is_delayed'] = (feat_df['status_clean'] == 'delayed').astype(int)
feat_df['hazard_bin'] = pd.cut(feat_df['weather_hazard_score'], bins=[0, 0.2, 0.4, 0.6, 1.0], labels=['Thấp (<0.2)', 'Trung bình (0.2-0.4)', 'Cao (0.4-0.6)', 'Cực đoan (>0.6)'])
hazard_delay = feat_df.groupby('hazard_bin', observed=False)['is_delayed'].mean() * 100

plt.figure(figsize=(8, 4))
hazard_delay.plot(kind='bar', color='#ea580c', edgecolor='black')
plt.title('Tỷ lệ Trễ chuyến (%) Theo Mức độ Nguy cơ Thời tiết')
plt.ylabel('Tỷ lệ trễ (%)')
plt.xticks(rotation=15)
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.tight_layout()
plt.show()
display(hazard_delay.to_frame('Tỷ lệ trễ (%)'))
